# 05 — Construção de um modelo baseline

**TCC UNIP 2026** · Etapa 5 de 9 do pipeline · Seção 4.5.5 do trabalho escrito.

**Objetivo:** validar o pipeline de ponta a ponta com uma ResNet50 de extrator congelado, treinada por poucas épocas (teste de fumaça).

**Entradas:**
- CSVs das partições e imagens processadas (notebook 03)

**Saídas:**
- `baseline_resnet50.pt`

Antes de executar: no Colab, selecione *Ambiente de execução → Alterar tipo de ambiente de execução → GPU* (necessário a partir do notebook 05). As saídas exibidas foram preservadas da execução original do notebook único.

In [ ]:
# Parâmetro: True usa o Google Drive para guardar os arquivos entre notebooks;
# False usa só o disco da sessão atual. Quando o notebook é executado pelo
# 00_executar_todos.ipynb, vale o valor definido lá.
if "USAR_DRIVE" not in globals():
    USAR_DRIVE = True


In [ ]:
# ============================================================
# PERSISTÊNCIA ENTRE NOTEBOOKS (Google Drive)
# ============================================================
# Cada notebook roda em uma sessão própria do Colab. Para que os arquivos gerados por um
# notebook fiquem disponíveis para os seguintes, a pasta /content/data é ligada a uma
# pasta no Google Drive. O valor de USAR_DRIVE é definido na célula anterior.
PASTA_DRIVE = "/content/drive/MyDrive/TCC_UNIP/data"

import os
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PASTA_DRIVE, exist_ok=True)
    if not os.path.islink("/content/data"):
        if os.path.isdir("/content/data"):
            if os.listdir("/content/data"):
                raise RuntimeError("/content/data já contém arquivos: mova-os para a pasta do Drive antes de continuar.")
            os.rmdir("/content/data")
        os.symlink(PASTA_DRIVE, "/content/data")
print("Pasta de dados:", os.path.realpath("/content/data"))


## Funções do pipeline de dados (reutilizadas do notebook 04)

In [ ]:
# [Célula reutilizada de 03_preprocessamento.ipynb]
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [ ]:
# [Célula reutilizada de 03_preprocessamento.ipynb]
SEED = 42
PROPORCAO_TREINO = 0.70
PROPORCAO_VAL = 0.15
PROPORCAO_TESTE = 0.15

# Mantido por compatibilidade com a estrutura original. Na versão corrigida, imagens sem
# lesion_id são mantidas e viram grupos próprios, salvo quando ligadas por pHash a duplicatas.
INCLUIR_SEM_LESION_ID = True

In [ ]:
# [Célula reutilizada de 04_pipeline_dados_pytorch.ipynb]
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
# [Célula reutilizada de 04_pipeline_dados_pytorch.ipynb]
SAIDA_BASE = "/content/data/processed"
MEAN_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET = [0.229, 0.224, 0.225]
BATCH_SIZE = 32
NUM_WORKERS = 2   # o Colab tem 2 CPUs; mais workers que isso só atrapalha (ver aviso do DataLoader)

# Modo de teste rápido (opcional): use p.ex. 0.05 para treinar/avaliar com ~5% das imagens
# e conferir se o notebook inteiro roda, antes de gastar horas de GPU. None = usa tudo.
FRACAO_DADOS_DEBUG = None

In [ ]:
# [Célula reutilizada de 04_pipeline_dados_pytorch.ipynb]
# ============================================================
# TRANSFORMS
# ============================================================
# ATENÇÃO (teste de cor): o ColorJitter do Albumentations, quando 'hue' não é informado, aplica
# uma rotação de MATIZ por padrão (nas versões 2.x: até ±0.5 do círculo cromático, i.e. qualquer cor).
# Isso muda a cor da lesão — um sinal clínico central (critério "C" do ABCDE) — em metade das imagens.
# Esta é a configuração da versão anterior e é mantida como variante "original" para comparação.
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.3),
    A.Normalize(mean=MEAN_IMAGENET, std=STD_IMAGENET),
    ToTensorV2(),
])

# Val/teste: SEM augmentation aleatório — só normalização, para avaliação justa e reprodutível
eval_transform = A.Compose([
    A.Normalize(mean=MEAN_IMAGENET, std=STD_IMAGENET),
    ToTensorV2(),
])

# ------------------------------------------------------------
# Variantes de augmentation de COR (usadas no "Teste de alterações de cor")
#   "original"         -> a configuração acima (brilho, contraste, saturação E matiz padrão do Albumentations)
#   "brilho_contraste" -> só brilho e contraste (±20%); saturação e matiz FIXOS (cor da lesão preservada)
#   "sem_cor"          -> nenhuma alteração de cor (só flips/rotações/escala)
# ------------------------------------------------------------
def construir_train_transform(variante_cor="original"):
    if variante_cor == "original":
        return train_transform
    if variante_cor == "brilho_contraste":
        cor = [A.ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2),
                             saturation=(1.0, 1.0), hue=(0.0, 0.0), p=0.5)]
    elif variante_cor == "sem_cor":
        cor = []
    else:
        raise ValueError(f"Variante de cor desconhecida: {variante_cor}")
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        *cor,
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.3),
        A.Normalize(mean=MEAN_IMAGENET, std=STD_IMAGENET),
        ToTensorV2(),
    ])

TRAIN_TRANSFORMS = {v: construir_train_transform(v)
                    for v in ["original", "brilho_contraste", "sem_cor"]}

In [ ]:
# [Célula reutilizada de 04_pipeline_dados_pytorch.ipynb]
# ============================================================
# DATASET
# ============================================================
from PIL import Image, ImageEnhance

def aplicar_perturbacao_cor(img_rgb, brilho=0.0, contraste=0.0, saturacao=0.0, matiz=0.0):
    """Perturbação DETERMINÍSTICA de cor (usada só na avaliação de robustez, nunca no treino).
    brilho/contraste/saturacao: variação relativa (0.2 = +20%, -0.2 = -20%).
    matiz: fração do círculo cromático (0.1 = 36 graus)."""
    pil = Image.fromarray(img_rgb)
    if brilho:
        pil = ImageEnhance.Brightness(pil).enhance(1.0 + brilho)
    if contraste:
        pil = ImageEnhance.Contrast(pil).enhance(1.0 + contraste)
    if saturacao:
        pil = ImageEnhance.Color(pil).enhance(1.0 + saturacao)
    if matiz:
        h, s, v = pil.convert("HSV").split()
        deslocamento = int(round(matiz * 255))
        h = h.point(lambda x: (x + deslocamento) % 256)
        pil = Image.merge("HSV", (h, s, v)).convert("RGB")
    return np.array(pil)


class DermatoDataset(Dataset):
    def __init__(self, dataframe, mapa_classes, transform=None, perturbacao=None):
        self.df = dataframe.reset_index(drop=True)
        self.mapa_classes = mapa_classes
        self.transform = transform
        self.perturbacao = perturbacao   # dict p/ aplicar_perturbacao_cor (None = sem perturbação)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        linha = self.df.iloc[idx]
        img = cv2.imread(linha["caminho_processado"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.perturbacao:
            img = aplicar_perturbacao_cor(img, **self.perturbacao)

        if self.transform:
            img = self.transform(image=img)["image"]

        rotulo = self.mapa_classes[linha["classe"]]
        return img, rotulo

In [ ]:
# [Célula reutilizada de 04_pipeline_dados_pytorch.ipynb]
# ============================================================
# CRIAÇÃO DOS DATALOADERS
# ============================================================
# Qual versão das imagens é usada por padrão: "com_dullrazor" (train.csv...) ou "sem_dullrazor".
# A célula de seleção da configuração final (mais adiante) atualiza este valor.
DADOS_ATIVOS = "com_dullrazor"

def caminho_csv_split(split, dados):
    sufixo = "" if dados == "com_dullrazor" else f"_{dados}"
    return f"{SAIDA_BASE}/{split}{sufixo}.csv"

def _ler_split(split, dados):
    df = pd.read_csv(caminho_csv_split(split, dados), low_memory=False)
    if FRACAO_DADOS_DEBUG:
        df = (df.groupby("classe", group_keys=False)
                .apply(lambda g: g.sample(n=max(2, int(len(g) * FRACAO_DADOS_DEBUG)),
                                          random_state=SEED)
                       if len(g) > 2 else g))
    return df

def criar_dataloaders(batch_size=BATCH_SIZE, dados=None, variante_cor="original"):
    dados = dados or DADOS_ATIVOS
    df_treino = _ler_split("train", dados)
    df_val = _ler_split("val", dados)
    df_teste = _ler_split("test", dados)

    classes_ordenadas = sorted(df_treino["classe"].unique())
    mapa_classes = {nome: idx for idx, nome in enumerate(classes_ordenadas)}
    print("Mapa de classes:", mapa_classes)
    print(f"Dados: {dados} | augmentation de cor: {variante_cor}")

    ds_treino = DermatoDataset(df_treino, mapa_classes, transform=TRAIN_TRANSFORMS[variante_cor])
    ds_val = DermatoDataset(df_val, mapa_classes, transform=eval_transform)
    ds_teste = DermatoDataset(df_teste, mapa_classes, transform=eval_transform)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(ds_treino, batch_size=batch_size, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=pin,
                               persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(ds_val, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=pin)
    test_loader = DataLoader(ds_teste, batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=pin)

    return train_loader, val_loader, test_loader, mapa_classes


def criar_loader_teste(dados=None, perturbacao=None, batch_size=BATCH_SIZE):
    """DataLoader de TESTE (sem augmentation), opcionalmente com perturbação determinística
    de cor — usado na avaliação de robustez a brilho/contraste."""
    dados = dados or DADOS_ATIVOS
    df_treino = _ler_split("train", dados)
    mapa_classes = {n: i for i, n in enumerate(sorted(df_treino["classe"].unique()))}
    ds = DermatoDataset(_ler_split("test", dados), mapa_classes,
                        transform=eval_transform, perturbacao=perturbacao)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=torch.cuda.is_available())
    return loader, mapa_classes

### BASELINE — ResNet50

BASELINE — ResNet50 com poucas épocas, só para validar o pipeline ponta a ponta
(dados -> modelo -> treino -> avaliação) antes de investir no treinamento completo
com fine-tuning em duas etapas.

Não espere um modelo bom aqui. O objetivo é confirmar que:
  1. Os DataLoaders entregam batches no formato certo.
  2. O modelo treina sem erro de shape/dispositivo/memória.
  3. A loss cai (nem que pouco) ao longo das épocas.
  4. A avaliação roda e produz métricas coerentes (nem que ruins).

Se tudo isso funcionar aqui, o próximo passo é o treinamento completo.

In [ ]:
import os
import time
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO BASELINE
# ============================================================
NUM_EPOCAS = 5          # poucas épocas mesmo — é só validação de pipeline
LEARNING_RATE = 1e-3    # taxa mais alta, já que só treinaremos a cabeça (camadas congeladas)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CAMINHO_CHECKPOINT = "/content/data/processed/baseline_resnet50.pt"

In [ ]:
def verificar_ambiente():
    print(f"Dispositivo: {DEVICE}")
    if DEVICE.type == "cpu":
        print("[AVISO] Nenhuma GPU detectada. Em Colab: Ambiente de execução > "
              "Alterar tipo de ambiente de execução > GPU. Treinar CNN em CPU "
              "é MUITO lento, mesmo para um baseline de 5 épocas.")
    else:
        print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def verificar_ambiente():
    print(f"Dispositivo: {DEVICE}")
    if DEVICE.type == "cpu":
        print("[AVISO] Nenhuma GPU detectada. Em Colab: Ambiente de execução > "
              "Alterar tipo de ambiente de execução > GPU. Treinar CNN em CPU "
              "é MUITO lento, mesmo para um baseline de 5 épocas.")
    else:
        print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CONSTRUÇÃO DO MODELO (transfer learning, cabeça descongelada só)
# ============================================================
def construir_modelo_baseline(num_classes):
    modelo = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Congela todas as camadas convolucionais (features já aprendidas no ImageNet)
    for param in modelo.parameters():
        param.requires_grad = False

    # Substitui e descongela só a última camada (classificador)
    modelo.fc = nn.Linear(modelo.fc.in_features, num_classes)

    return modelo.to(DEVICE)

In [ ]:
# ============================================================
# CLASS WEIGHTS (dataset desbalanceado, como já mapeamos na EDA)
# ============================================================
def calcular_pesos_classe(train_loader, mapa_classes):
    todos_rotulos = []
    for _, rotulos in train_loader:
        todos_rotulos.extend(rotulos.tolist())

    classes_presentes = sorted(set(todos_rotulos))
    pesos = compute_class_weight(
        class_weight="balanced",
        classes=np.array(classes_presentes),
        y=np.array(todos_rotulos),
    )
    # Garante que o vetor de pesos tenha o tamanho certo mesmo se alguma
    # classe rara não aparecer em nenhum batch do treino (caso extremo)
    vetor_pesos = np.ones(len(mapa_classes))
    for classe_idx, peso in zip(classes_presentes, pesos):
        vetor_pesos[classe_idx] = peso

    print("Pesos por classe (compensa o desbalanceamento):")
    nomes_ordenados = sorted(mapa_classes, key=lambda k: mapa_classes[k])
    for nome, peso in zip(nomes_ordenados, vetor_pesos):
        print(f"  {nome}: {peso:.3f}")

    return torch.tensor(vetor_pesos, dtype=torch.float32).to(DEVICE)

In [ ]:
# ============================================================
# LOOP DE TREINO E VALIDAÇÃO
# ============================================================
def treinar_epoca(modelo, loader, criterio, otimizador):
    modelo.train()
    perda_total = 0.0
    acertos = 0
    total = 0

    for imagens, rotulos in loader:
        imagens, rotulos = imagens.to(DEVICE), rotulos.to(DEVICE)

        otimizador.zero_grad()
        saidas = modelo(imagens)
        perda = criterio(saidas, rotulos)
        perda.backward()
        otimizador.step()

        perda_total += perda.item() * imagens.size(0)
        acertos += (saidas.argmax(dim=1) == rotulos).sum().item()
        total += imagens.size(0)

    return perda_total / total, acertos / total

In [ ]:
def validar_epoca(modelo, loader, criterio):
    modelo.eval()
    perda_total = 0.0
    acertos = 0
    total = 0

    with torch.no_grad():
        for imagens, rotulos in loader:
            imagens, rotulos = imagens.to(DEVICE), rotulos.to(DEVICE)
            saidas = modelo(imagens)
            perda = criterio(saidas, rotulos)

            perda_total += perda.item() * imagens.size(0)
            acertos += (saidas.argmax(dim=1) == rotulos).sum().item()
            total += imagens.size(0)

    return perda_total / total, acertos / total

In [ ]:
# ============================================================
# AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
# ============================================================
def avaliar_teste(modelo, test_loader, mapa_classes):
    modelo.eval()
    todos_rotulos = []
    todas_preds = []

    with torch.no_grad():
        for imagens, rotulos in test_loader:
            imagens = imagens.to(DEVICE)
            saidas = modelo(imagens)
            preds = saidas.argmax(dim=1).cpu().numpy()

            todos_rotulos.extend(rotulos.numpy())
            todas_preds.extend(preds)

    nomes_classes = sorted(mapa_classes, key=lambda k: mapa_classes[k])

    print("\n--- Matriz de confusão (baseline, teste) ---")
    print(confusion_matrix(todos_rotulos, todas_preds))

    print("\n--- Relatório de classificação (baseline, teste) ---")
    print(classification_report(todos_rotulos, todas_preds, target_names=nomes_classes, zero_division=0))

In [ ]:
# ============================================================
# EXECUÇÃO
# ============================================================
if __name__ == "__main__":
    verificar_ambiente()

    print("\n" + "=" * 60)
    print("Carregando DataLoaders")
    print("=" * 60)
    train_loader, val_loader, test_loader, mapa_classes = criar_dataloaders()
    num_classes = len(mapa_classes)
    print(f"Número de classes: {num_classes}")

    print("\n" + "=" * 60)
    print("Construindo modelo baseline (ResNet50, cabeça descongelada)")
    print("=" * 60)
    modelo = construir_modelo_baseline(num_classes)

    print("\n" + "=" * 60)
    print("Calculando pesos de classe (dataset desbalanceado)")
    print("=" * 60)
    pesos_classe = calcular_pesos_classe(train_loader, mapa_classes)
    criterio = nn.CrossEntropyLoss(weight=pesos_classe)

    otimizador = torch.optim.Adam(
        filter(lambda p: p.requires_grad, modelo.parameters()),
        lr=LEARNING_RATE
    )

    print("\n" + "=" * 60)
    print(f"Treinando baseline por {NUM_EPOCAS} épocas")
    print("=" * 60)

    for epoca in range(1, NUM_EPOCAS + 1):
        inicio = time.time()

        perda_treino, acc_treino = treinar_epoca(modelo, train_loader, criterio, otimizador)
        perda_val, acc_val = validar_epoca(modelo, val_loader, criterio)

        duracao = time.time() - inicio
        print(f"Época {epoca}/{NUM_EPOCAS} "
              f"| Treino: loss={perda_treino:.4f} acc={acc_treino:.4f} "
              f"| Val: loss={perda_val:.4f} acc={acc_val:.4f} "
              f"| Tempo: {duracao:.1f}s")

    torch.save(modelo.state_dict(), CAMINHO_CHECKPOINT)
    print(f"\nCheckpoint salvo em: {CAMINHO_CHECKPOINT}")

    print("\n" + "=" * 60)
    print("Avaliação final no conjunto de TESTE (só para conferência do pipeline)")
    print("=" * 60)
    avaliar_teste(modelo, test_loader, mapa_classes)

    print("\n" + "=" * 60)
    print("CHECKLIST DE VALIDAÇÃO DO PIPELINE")
    print("=" * 60)
    print("[ ] A loss de treino caiu ao longo das épocas? (confira o log acima)")
    print("[ ] Não houve erro de shape/memória/dispositivo?")
    print("[ ] A matriz de confusão tem valores em quase todas as classes")
    print("    (mesmo que a acurácia esteja baixa)?")
    print("[ ] Se sim para tudo: pipeline validado, pronto para o treinamento completo.")

Dispositivo: cuda
GPU: Tesla T4

Carregando DataLoaders
Mapa de classes: {'carcinoma_basocelular': 0, 'ceratose_actinica': 1, 'ceratose_seborreica': 2, 'dermatofibroma': 3, 'lesao_vascular': 4, 'melanoma': 5, 'nevo_melanocitico': 6}
Dados: com_dullrazor | augmentation de cor: original
Número de classes: 7

Construindo modelo baseline (ResNet50, cabeça descongelada)

Calculando pesos de classe (dataset desbalanceado)
Pesos por classe (compensa o desbalanceamento):
  carcinoma_basocelular: 1.028
  ceratose_actinica: 3.845
  ceratose_seborreica: 2.478
  dermatofibroma: 13.274
  lesao_vascular: 13.986
  melanoma: 0.724
  nevo_melanocitico: 0.261

Treinando baseline por 5 épocas
Época 1/5 | Treino: loss=1.5263 acc=0.5113 | Val: loss=1.1396 acc=0.5517 | Tempo: 72.1s
Época 2/5 | Treino: loss=1.3345 acc=0.5375 | Val: loss=1.0261 acc=0.6283 | Tempo: 71.6s
Época 3/5 | Treino: loss=1.2520 acc=0.5576 | Val: loss=1.1358 acc=0.5772 | Tempo: 71.7s
Época 4/5 | Treino: loss=1.2483 acc=0.5492 | Val: los